# Ensemble Methods

## Overview
This notebook combines predictions from multiple trained models (ResNet50, 
EfficientNet, and the two-stage Stage 1/Stage 2 models) to test whether 
ensemble methods improve upon individual model performance.

## Approaches Tested
1. **Equal ensemble** - average probabilities from all models equally
2. **Weighted ensemble** - give Stage 2 more weight for tumor classification
3. **Sensor fusion** - trust the best model exclusively for glioma, 
   average for other classes

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from google.colab import drive

In [2]:
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

train_dir = "/content/drive/MyDrive/brain-tumor-data/Training"
test_dir = "/content/drive/MyDrive/brain-tumor-data/Testing"

classes = ["glioma", "meningioma", "notumor", "pituitary"]
tumor_classes = ["glioma", "meningioma", "pituitary"]
num_classes = len(classes)
class_to_idx = {cls: i for i, cls in enumerate(classes)}
tumor_label = {"glioma": 0, "meningioma": 1, "pituitary": 2}
tumor_idx_to_class = {0: "glioma", 1: "meningioma", 2: "pituitary"}
binary_label = {"glioma": 1, "meningioma": 1, "notumor": 0, "pituitary": 1}

IMG_SIZE = 224
BATCH_SIZE = 32

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("Setup done!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Setup done!


In [3]:
def build_resnet(num_classes):
    model = models.resnet50(weights=None)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

def build_efficientnet(num_classes):
    model = models.efficientnet_b0(weights=None)
    for param in model.parameters():
        param.requires_grad = False
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model.to(device)

# Load all models
resnet_model = build_resnet(4)
resnet_model.load_state_dict(torch.load("/content/drive/MyDrive/brain-tumor-data/brain_tumor_resnet50.pth", map_location=device))
resnet_model.eval()
print("ResNet50 loaded!")

efficientnet_model = build_efficientnet(4)
efficientnet_model.load_state_dict(torch.load("/content/drive/MyDrive/brain-tumor-data/efficientnet_model.pth", map_location=device))
efficientnet_model.eval()
print("EfficientNet loaded!")

stage1_model = build_resnet(2)
stage1_model.load_state_dict(torch.load("/content/drive/MyDrive/brain-tumor-data/stage1_model.pth", map_location=device))
stage1_model.eval()
print("Stage 1 loaded!")

stage2_model = build_resnet(3)
stage2_model.load_state_dict(torch.load("/content/drive/MyDrive/brain-tumor-data/stage2_model.pth", map_location=device))
stage2_model.eval()
print("Stage 2 loaded!")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 209MB/s]


ResNet50 loaded!
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 115MB/s] 


EfficientNet loaded!
Stage 1 loaded!
Stage 2 loaded!


## Equal Ensemble

Combines Stage 1 (tumor detection) with an average of ResNet50, 
EfficientNet, and Stage 2 probabilities for tumor type classification.

**Result:** 87.75% accuracy - the best result so far at this point in 
the project, with notable improvement in meningioma recall (85.5%).

In [4]:
def ensemble_predict(img_path):
    # Load image
    img = Image.open(img_path).convert("RGB")
    tensor = test_transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Stage 1: tumor or not?
        s1_out = torch.softmax(stage1_model(tensor), dim=1)
        notumor_prob = s1_out[0, 0].item()
       
        
        if s1_out.argmax().item() == 0:
            # No tumor detected - return directly
            probs = [0.0, 0.0, notumor_prob, 0.0]
            return 2, probs  # notumor = index 2
        
        # Tumor detected - run ensemble on tumor type
        # ResNet50 probabilities
        resnet_out = torch.softmax(resnet_model(tensor), dim=1)[0]
        resnet_tumor_probs = torch.tensor([
            resnet_out[0],  # glioma
            resnet_out[1],  # meningioma
            resnet_out[3]   # pituitary
        ]).to(device)
        
        # EfficientNet probabilities
        eff_out = torch.softmax(efficientnet_model(tensor), dim=1)[0]
        eff_tumor_probs = torch.tensor([
            eff_out[0],  # glioma
            eff_out[1],  # meningioma
            eff_out[3]   # pituitary
        ]).to(device)
        
        # Stage 2 probabilities
        s2_out = torch.softmax(stage2_model(tensor), dim=1)[0]
        
        # Average all three
        avg_probs = (resnet_tumor_probs + eff_tumor_probs + s2_out) / 3
        tumor_pred = avg_probs.argmax().item()
        predicted_class = tumor_idx_to_class[tumor_pred]
        
        # Build final 4-class probabilities
        final_probs = [
            avg_probs[0].item(),  # glioma
            avg_probs[1].item(),  # meningioma
            notumor_prob,         # notumor
            avg_probs[2].item()   # pituitary
        ]
        
        return class_to_idx[predicted_class], final_probs

print("Ensemble pipeline ready!")

Ensemble pipeline ready!


In [7]:
all_preds = []
all_labels = []

for cls in classes:
    cls_path = os.path.join(test_dir, cls)
    true_label = class_to_idx[cls]
    print(f"Processing {cls}...")
    
    for img_file in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img_file)
        pred, probs = ensemble_predict(img_path)
        all_preds.append(pred)
        all_labels.append(true_label)
    
    cls_preds = all_preds[-400:]
    cls_correct = sum(p == true_label for p in cls_preds)
    print(f"  {cls}: {cls_correct}/400 correct ({100*cls_correct/400:.1f}%)")

accuracy = 100 * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"\nEnsemble Test Accuracy: {accuracy:.2f}%")
print(classification_report(all_labels, all_preds, target_names=classes))

Processing glioma...
  glioma: 287/400 correct (71.8%)
Processing meningioma...
  meningioma: 342/400 correct (85.5%)
Processing notumor...
  notumor: 389/400 correct (97.2%)
Processing pituitary...
  pituitary: 386/400 correct (96.5%)

Ensemble Test Accuracy: 87.75%
              precision    recall  f1-score   support

      glioma       0.93      0.72      0.81       400
  meningioma       0.77      0.85      0.81       400
     notumor       0.90      0.97      0.94       400
   pituitary       0.93      0.96      0.95       400

    accuracy                           0.88      1600
   macro avg       0.88      0.88      0.88      1600
weighted avg       0.88      0.88      0.88      1600



## Weighted Ensemble

Tests whether giving more weight to Stage 2 (the strongest model for 
glioma) improves results.

**Result:** 87.00% accuracy - slightly worse than equal weighting. 
Increasing Stage 2's weight improved glioma slightly (72.8%) but hurt 
meningioma performance (81.8%), showing that blanket reweighting affects 
all classes, not just the intended one.

In [5]:
def weighted_ensemble_predict(img_path):
    img = Image.open(img_path).convert("RGB")
    tensor = test_transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Stage 1: tumor or not?
        s1_out = torch.softmax(stage1_model(tensor), dim=1)
        notumor_prob = s1_out[0, 0].item()
        tumor_prob = s1_out[0, 1].item()
        
        if s1_out.argmax().item() == 0:
            probs = [0.0, 0.0, notumor_prob, 0.0]
            return 2, probs
        
        # ResNet50 probabilities
        resnet_out = torch.softmax(resnet_model(tensor), dim=1)[0]
        resnet_tumor_probs = torch.tensor([
            resnet_out[0],
            resnet_out[1],
            resnet_out[3]
        ]).to(device)
        
        # EfficientNet probabilities
        eff_out = torch.softmax(efficientnet_model(tensor), dim=1)[0]
        eff_tumor_probs = torch.tensor([
            eff_out[0],
            eff_out[1],
            eff_out[3]
        ]).to(device)
        
        # Stage 2 probabilities
        s2_out = torch.softmax(stage2_model(tensor), dim=1)[0]
        
        # Weighted average - trust stage2 more for tumor classification
        w_resnet = 0.25
        w_eff = 0.25
        w_stage2 = 0.50
        
        avg_probs = (w_resnet * resnet_tumor_probs + 
                     w_eff * eff_tumor_probs + 
                     w_stage2 * s2_out)
        
        tumor_pred = avg_probs.argmax().item()
        predicted_class = tumor_idx_to_class[tumor_pred]
        
        final_probs = [
            avg_probs[0].item(),
            avg_probs[1].item(),
            notumor_prob,
            avg_probs[2].item()
        ]
        
        return class_to_idx[predicted_class], final_probs

print("Weighted ensemble ready!")

Weighted ensemble ready!


In [6]:
all_preds = []
all_labels = []

for cls in classes:
    cls_path = os.path.join(test_dir, cls)
    true_label = class_to_idx[cls]
    print(f"Processing {cls}...")
    
    for img_file in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img_file)
        pred, probs = weighted_ensemble_predict(img_path)
        all_preds.append(pred)
        all_labels.append(true_label)
    
    cls_preds = all_preds[-400:]
    cls_correct = sum(p == true_label for p in cls_preds)
    print(f"  {cls}: {cls_correct}/400 correct ({100*cls_correct/400:.1f}%)")

accuracy = 100 * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"\nWeighted Ensemble Test Accuracy: {accuracy:.2f}%")
print(classification_report(all_labels, all_preds, target_names=classes))

Processing glioma...
  glioma: 291/400 correct (72.8%)
Processing meningioma...
  meningioma: 327/400 correct (81.8%)
Processing notumor...
  notumor: 389/400 correct (97.2%)
Processing pituitary...
  pituitary: 385/400 correct (96.2%)

Weighted Ensemble Test Accuracy: 87.00%
              precision    recall  f1-score   support

      glioma       0.89      0.73      0.80       400
  meningioma       0.77      0.82      0.79       400
     notumor       0.90      0.97      0.94       400
   pituitary       0.93      0.96      0.94       400

    accuracy                           0.87      1600
   macro avg       0.87      0.87      0.87      1600
weighted avg       0.87      0.87      0.87      1600



## Sensor Fusion

Inspired by sensor fusion in autonomous systems, this approach trusts 
Stage 2 exclusively for glioma (its strongest class) while averaging 
all models for meningioma and pituitary.

**Result:** 86.69% accuracy - improved glioma recall (74%) but lower 
overall accuracy than equal ensemble. This shows that excluding other 
models' votes entirely discards useful information, even when one 
model is statistically best on average for that class.

In [5]:
def sensor_fusion_predict(img_path):
    img = Image.open(img_path).convert("RGB")
    tensor = test_transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Stage 1: tumor or not?
        s1_out = torch.softmax(stage1_model(tensor), dim=1)
        notumor_prob = s1_out[0, 0].item()
        
        if s1_out.argmax().item() == 0:
            probs = [0.0, 0.0, notumor_prob, 0.0]
            return 2, probs
        
        # ResNet50 probabilities
        resnet_out = torch.softmax(resnet_model(tensor), dim=1)[0]
        resnet_tumor_probs = torch.tensor([
            resnet_out[0],
            resnet_out[1],
            resnet_out[3]
        ]).to(device)
        
        # EfficientNet probabilities
        eff_out = torch.softmax(efficientnet_model(tensor), dim=1)[0]
        eff_tumor_probs = torch.tensor([
            eff_out[0],
            eff_out[1],
            eff_out[3]
        ]).to(device)
        
        # Stage 2 probabilities
        s2_out = torch.softmax(stage2_model(tensor), dim=1)[0]
        
        # Sensor fusion:
        # Glioma     → Stage 2 only
        # Meningioma → average all three
        # Pituitary  → average all three
        glioma_prob = s2_out[0]
        meningioma_prob = (resnet_tumor_probs[1] + eff_tumor_probs[1] + s2_out[1]) / 3
        pituitary_prob = (resnet_tumor_probs[2] + eff_tumor_probs[2] + s2_out[2]) / 3
        
        avg_probs = torch.tensor([
            glioma_prob.item(),
            meningioma_prob.item(),
            pituitary_prob.item()
        ]).to(device)
        
        tumor_pred = avg_probs.argmax().item()
        predicted_class = tumor_idx_to_class[tumor_pred]
        
        final_probs = [
            avg_probs[0].item(),
            avg_probs[1].item(),
            notumor_prob,
            avg_probs[2].item()
        ]
        
        return class_to_idx[predicted_class], final_probs

print("Sensor fusion ready!")

Sensor fusion ready!


In [6]:
all_preds = []
all_labels = []

for cls in classes:
    cls_path = os.path.join(test_dir, cls)
    true_label = class_to_idx[cls]
    print(f"Processing {cls}...")
    
    for img_file in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img_file)
        pred, probs = sensor_fusion_predict(img_path)
        all_preds.append(pred)
        all_labels.append(true_label)
    
    cls_preds = all_preds[-400:]
    cls_correct = sum(p == true_label for p in cls_preds)
    print(f"  {cls}: {cls_correct}/400 correct ({100*cls_correct/400:.1f}%)")

accuracy = 100 * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"\nSensor Fusion Test Accuracy: {accuracy:.2f}%")
print(classification_report(all_labels, all_preds, target_names=classes))

Processing glioma...
  glioma: 298/400 correct (74.5%)
Processing meningioma...
  meningioma: 319/400 correct (79.8%)
Processing notumor...
  notumor: 389/400 correct (97.2%)
Processing pituitary...
  pituitary: 381/400 correct (95.2%)

Sensor Fusion Test Accuracy: 86.69%
              precision    recall  f1-score   support

      glioma       0.85      0.74      0.79       400
  meningioma       0.78      0.80      0.79       400
     notumor       0.90      0.97      0.94       400
   pituitary       0.93      0.95      0.94       400

    accuracy                           0.87      1600
   macro avg       0.87      0.87      0.87      1600
weighted avg       0.87      0.87      0.87      1600

